# Customer Retention Intelligence Platform

## 01 – Data Audit

### Objective

Evaluate the raw customer dataset before preprocessing or modeling.

This notebook examines:
- Dataset dimensions and schema
- Data types
- Missing values
- Duplicate records
- Identifier uniqueness
- Target distribution
- Numerical and categorical features
- Potential data quality issues
- Potential sources of target leakage

In [1]:
from pathlib import Path
from src.data.pipeline import run_data_pipeline

import pandas as pd

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

data_path = project_root / "data" / "raw" / "WA_Fn-UseC_-Telco-Customer-Churn.csv"

df = run_data_pipeline(data_path)

In [2]:
df.shape

(7043, 21)

In [5]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


IDENTIFIER
    ↓
Who is this customer?
customerID

CUSTOMER CHARACTERISTICS
    ↓
Who/what type of customer are they?
gender, SeniorCitizen, Partner, Dependents

RELATIONSHIP
    ↓
How long / under what agreement have they been a customer?
tenure, Contract

SERVICES
    ↓
What products are they using?
PhoneService, MultipleLines, InternetService, OnlineSecurity, DeviceProtection, TechSupport, StreamingTV, StreamingMovies

FINANCIAL
    ↓
What are they paying?
MonthlyCharges, TotalCharges

TARGET
    ↓
Did they churn?
Churn

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [7]:
df.columns.to_list()

['customerID',
 'gender',
 'SeniorCitizen',
 'Partner',
 'Dependents',
 'tenure',
 'PhoneService',
 'MultipleLines',
 'InternetService',
 'OnlineSecurity',
 'OnlineBackup',
 'DeviceProtection',
 'TechSupport',
 'StreamingTV',
 'StreamingMovies',
 'Contract',
 'PaperlessBilling',
 'PaymentMethod',
 'MonthlyCharges',
 'TotalCharges',
 'Churn']

In [8]:
df['TotalCharges'].head(20)

0       29.85
1      1889.5
2      108.15
3     1840.75
4      151.65
5       820.5
6      1949.4
7       301.9
8     3046.05
9     3487.95
10     587.45
11      326.8
12     5681.1
13     5036.3
14    2686.05
15    7895.15
16    1022.95
17    7382.25
18     528.35
19     1862.9
Name: TotalCharges, dtype: str

In [9]:
df['TotalCharges'].unique()

<StringArray>
[  '29.85',  '1889.5',  '108.15', '1840.75',  '151.65',   '820.5',  '1949.4',
   '301.9', '3046.05', '3487.95',
 ...
 '2625.25', '6886.25',  '1495.1',   '743.3',  '1419.4',  '1990.5',  '7362.9',
  '346.45',   '306.6',  '6844.5']
Length: 6531, dtype: str

In [10]:
total_charges_numeric = pd.to_numeric(
    df['TotalCharges'],
    errors='coerce'
)

df[total_charges_numeric.isna()][
    ['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Churn']
]

,customerID,tenure,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,0,52.55,,No
753,3115-CZMZD,0,20.25,,No
936,5709-LVOEQ,0,80.85,,No
1082,4367-NUYAO,0,25.75,,No
1340,1371-DWPAZ,0,56.05,,No
3331,7644-OMVMY,0,19.85,,No
3826,3213-VVOLG,0,25.35,,No
4380,2520-SGTTA,0,20.00,,No
5218,2923-ARZLG,0,19.70,,No
6670,4075-WKNIU,0,73.35,,No


It looks as these are new customers which is why there is not a historcal value in TotalCharges.

In [11]:
df.isna().sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

In [12]:
df['Churn'].value_counts()

Churn
No     5174
Yes    1869
Name: count, dtype: int64

In [13]:
df['Churn'].value_counts(normalize=True)

Churn
No     0.73463
Yes    0.26537
Name: proportion, dtype: float64

In [14]:
df['customerID'].nunique()

7043

In [15]:
df.duplicated().sum()

np.int64(0)

In [16]:
categorical_columns = df.select_dtypes(include=['object', 'string']).columns

for column in categorical_columns:
    print(f"\n{column}")
    print(df[column].value_counts(dropna=False))


customerID
customerID
7590-VHVEG    1
5575-GNVDE    1
3668-QPYBK    1
7795-CFOCW    1
9237-HQITU    1
             ..
6840-RESVB    1
2234-XADUH    1
4801-JZAZL    1
8361-LTMKD    1
3186-AJIEK    1
Name: count, Length: 7043, dtype: int64

gender
gender
Male      3555
Female    3488
Name: count, dtype: int64

Partner
Partner
No     3641
Yes    3402
Name: count, dtype: int64

Dependents
Dependents
No     4933
Yes    2110
Name: count, dtype: int64

PhoneService
PhoneService
Yes    6361
No      682
Name: count, dtype: int64

MultipleLines
MultipleLines
No                  3390
Yes                 2971
No phone service     682
Name: count, dtype: int64

InternetService
InternetService
Fiber optic    3096
DSL            2421
No             1526
Name: count, dtype: int64

OnlineSecurity
OnlineSecurity
No                     3498
Yes                    2019
No internet service    1526
Name: count, dtype: int64

OnlineBackup
OnlineBackup
No                     3088
Yes                    2429


In [17]:
# create a temporary diagnostic version of total charges rather than modify the raw dataset
numeric_audit = df.copy()

numeric_audit['TotalCharges'] = pd.to_numeric(
    numeric_audit['TotalCharges'],
    errors='coerce'
)

In [18]:
numeric_audit[
    ['tenure', 'MonthlyCharges', 'TotalCharges']
].describe()

,tenure,MonthlyCharges,TotalCharges
count,7043.000000,7043.000000,7032.000000
mean,32.371149,64.761692,2283.300441
std,24.559481,30.090047,2266.771362
min,0.000000,18.250000,18.800000
25%,9.000000,35.500000,401.450000
50%,29.000000,70.350000,1397.475000
75%,55.000000,89.850000,3794.737500
max,72.000000,118.750000,8684.800000


In [19]:
# verify hypothesis about the 11 missing values
numeric_audit[
    numeric_audit['TotalCharges'].isna()
]['tenure'].value_counts()

tenure
0    11
Name: count, dtype: int64

In [20]:
# create a temporary calculation of the estimated total charges based on tenure and MonthlyCharges
numeric_audit['estimated_total_charges'] = (
    numeric_audit['tenure']
    * numeric_audit['MonthlyCharges']
)

In [21]:
numeric_audit[
    [
        'tenure',
        'MonthlyCharges',
        'TotalCharges',
        'estimated_total_charges'
    ]
].head(20)

,tenure,MonthlyCharges,TotalCharges,estimated_total_charges
0,1,29.85,29.85,29.85
1,34,56.95,1889.50,1936.30
2,2,53.85,108.15,107.70
3,45,42.30,1840.75,1903.50
4,2,70.70,151.65,141.40
5,8,99.65,820.50,797.20
6,22,89.10,1949.40,1960.20
7,10,29.75,301.90,297.50
8,28,104.80,3046.05,2934.40
9,62,56.15,3487.95,3481.30


In [25]:
numeric_audit[
    [
        'tenure',
        'MonthlyCharges',
        'TotalCharges',
        'estimated_total_charges'
    ]
].describe()

,tenure,MonthlyCharges,TotalCharges,estimated_total_charges
count,7043.000000,7043.000000,7032.000000,7043.000000
mean,32.371149,64.761692,2283.300441,2279.581350
std,24.559481,30.090047,2266.771362,2264.729447
min,0.000000,18.250000,18.800000,0.000000
25%,9.000000,35.500000,401.450000,394.000000
50%,29.000000,70.350000,1397.475000,1393.600000
75%,55.000000,89.850000,3794.737500,3786.100000
max,72.000000,118.750000,8684.800000,8550.000000


In [24]:
numeric_audit[numeric_audit["TotalCharges"].isna()]

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,estimated_total_charges
488,4472-LVYGI,Female,0,Yes,Yes,0,No,No phone service,DSL,Yes,...,Yes,Yes,No,Two year,Yes,Bank transfer (automatic),52.55,NaN,No,0.0
753,3115-CZMZD,Male,0,No,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.25,NaN,No,0.0
936,5709-LVOEQ,Female,0,Yes,Yes,0,Yes,No,DSL,Yes,...,No,Yes,Yes,Two year,No,Mailed check,80.85,NaN,No,0.0
1082,4367-NUYAO,Male,0,Yes,Yes,0,Yes,Yes,No,No internet service,...,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.75,NaN,No,0.0
1340,1371-DWPAZ,Female,0,Yes,Yes,0,No,No phone service,DSL,Yes,...,Yes,Yes,No,Two year,No,Credit card (automatic),56.05,NaN,No,0.0
3331,7644-OMVMY,Male,0,Yes,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,Two year,No,Mailed check,19.85,NaN,No,0.0
3826,3213-VVOLG,Male,0,Yes,Yes,0,Yes,Yes,No,No internet service,...,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.35,NaN,No,0.0
4380,2520-SGTTA,Female,0,Yes,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.00,NaN,No,0.0
5218,2923-ARZLG,Male,0,Yes,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,One year,Yes,Mailed check,19.70,NaN,No,0.0
6670,4075-WKNIU,Female,0,Yes,Yes,0,Yes,Yes,DSL,No,...,Yes,Yes,No,Two year,No,Mailed check,73.35,NaN,No,0.0


In [28]:
# Create a churn-rate table for categorical variables
# Start with Contract:
contract_churn = pd.crosstab(
    df["Contract"],
    df["Churn"],
    normalize="index"
)

contract_churn

Churn,No,Yes
Contract,,
Month-to-month,0.572903,0.427097
One year,0.887305,0.112695
Two year,0.971681,0.028319


In [29]:
# Then investigate InternetService:
internet_churn = pd.crosstab(
    df["InternetService"],
    df["Churn"],
    normalize="index"
)

internet_churn

Churn,No,Yes
InternetService,,
DSL,0.810409,0.189591
Fiber optic,0.581072,0.418928
No,0.925950,0.074050


In [30]:
# And PaymentMethod:
payment_churn = pd.crosstab(
    df["PaymentMethod"],
    df["Churn"],
    normalize="index"
)

payment_churn

Churn,No,Yes
PaymentMethod,,
Bank transfer (automatic),0.832902,0.167098
Credit card (automatic),0.847569,0.152431
Electronic check,0.547146,0.452854
Mailed check,0.808933,0.191067


In [33]:
# Examine Tenure
df.groupby("Churn")["tenure"].describe()

,count,mean,std,min,25%,50%,75%,max
Churn,,,,,,,,
No,5174.0,37.569965,24.113777,0.0,15.0,38.0,61.0,72.0
Yes,1869.0,17.979133,19.531123,1.0,2.0,10.0,29.0,72.0


In [34]:
# Examine MonthlyCharges
df.groupby("Churn")["MonthlyCharges"].describe()

,count,mean,std,min,25%,50%,75%,max
Churn,,,,,,,,
No,5174.0,61.265124,31.092648,18.25,25.10,64.425,88.4,118.75
Yes,1869.0,74.441332,24.666053,18.85,56.15,79.650,94.2,118.35


## Data Audit Summary

### Dataset
- 7,043 customer records
- 21 original variables
- 7,043 unique customer IDs
- 0 duplicate records

### Data Quality
- TotalCharges was loaded as a string rather than a numeric variable.
- 11 TotalCharges values are blank.
- All 11 affected customers have tenure = 0, suggesting they are newly acquired customers rather than corrupted records.
- No other obvious missing-value issues were identified.

### Target
- 5,174 customers (73.46%) did not churn.
- 1,869 customers (26.54%) churned.
- The target is moderately imbalanced, making accuracy alone an inadequate evaluation metric.

### Initial Business Observations
* Customers in one year and two year contracts appear to churn significantly less than customers on a month-to-month basis.
* Customers with fiber optic internet service appear substantially more likely to churn than those with DSL or no internet service.
* Customers who pay by electronic check appear substantially more likely to churn than customers who pay by any other payment method.
* It appears as though the average tenure for customers who churn is 17.98 as opposed to 37.57.
* It also appears that the average monthly charges for customers who churn is 74.44 as opposed to 61.27

### Initial Cleaning Decisions
- Convert TotalCharges to numeric.
- Preserve customers with tenure = 0 rather than dropping them.
- Treat blank TotalCharges for zero-tenure customers as 0.
- Retain customerID for customer tracking but exclude it as a predictive feature.
- Preserve "No internet service" and "No phone service" as distinct categorical states.
- Treat SeniorCitizen conceptually as a binary categorical indicator.